In [1]:
import torch
import torch.nn as nn
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def positional_encoding(seq_len, d_model):
    positions = torch.arange(seq_len)[:, np.newaxis]
    angles = torch.arange(d_model)[np.newaxis, :] / d_model
    angles = angles * np.power(10000, -2 * (angles // 2) / d_model)
    encodings = torch.zeros((seq_len, d_model))
    encodings[:, 0::2] = torch.sin(positions * angles[:, 0::2])
    encodings[:, 1::2] = torch.cos(positions * angles[:, 1::2])
    return encodings.unsqueeze(0).to(device)

def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.shape[-1]
    scores = torch.matmul(query, np.swapaxes(key, -2, -1)) / np.sqrt(d_k)
    if mask is not None:
        scores = torch.where(mask == 0, -1e9, scores)
    attention_weights = softmax(scores)
    return torch.matmul(attention_weights, value), attention_weights

def softmax(x, axis=-1):
    #exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    torch.softmax(x, dim=-1).to(device)
    return torch.softmax(x, dim=-1).to(device)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        
        self.d_k = d_model 
        self.d_model = d_model
        d_model%num_heads
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.W_v = nn.Linear(d_model, d_model)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        print("value")
        self.W_ff = nn.Linear(d_model, d_model)
        

    
    def forward(self, query, key, value, mask=None):
        #split heads
        b,t,_ = query.size()
        query = query.view(b,t, self.num_heads, self.d_k).transpose(1, 2)
        b,t,_ = key.size()
        key = key.view(b,t, self.num_heads, self.d_k).transpose(1, 2)
        b,t,_ = value.size()
        value = value.view(b,t, self.num_heads, self.d_k).transpose(1, 2)
        
        attention_output, _ = scaled_dot_product_attention(query, key, value, mask)
        attention_output = attention_output.transpose(1, 2).contiguous().view(query.size(0), -1, self.d_model)
        return self.W_ff(attention_output)

class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.dropout  = nn.Dropout(0.2)  #have to check
        self.fc1 = nn.Linear(d_model, d_ff)
        self.dropout1  = nn.Dropout(0.1)  #have to check
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

def create_padding_mask(seq):
    return (seq == 0).unsqueeze(1).unsqueeze(2)

def create_look_ahead_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).type(torch.bool)
    return mask

In [2]:
import torch
import torch.nn as nn
#from utils import MultiHeadAttention, PositionWiseFeedForward, positional_encoding

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.dropout1 = nn.Dropout(dropout) # check
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attention(x, x, x, mask)
        drop_attn = self.dropout(attn_output)
        x = self.norm1(x + drop_attn)
        ff_output = self.feed_forward(x)
        drop_ff = self.dropout(ff_output)
        return self.norm2(x + drop_ff)


# class Encoder(nn.Module):
#     def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout):
#         super(Encoder, self).__init__()
#         self.d_model = d_model
#         self.embedding = nn.Embedding(vocab_size, d_model)
#         self.positional_encoding = positional_encoding(max_seq_length, d_model)
#         self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
#         self.dropout = nn.Dropout(dropout)

#     def forward(self, x, mask):
#         seq_length = x.size(1)
#         x = self.embedding(x) * math.sqrt(self.d_model)
#         x = x + self.positional_encoding[:, :seq_length, :].to(x.device)
#         x = self.dropout(x)

#         for layer in self.layers:
#             x = layer(x, mask)

#         return x
    
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout):
        super(Encoder, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.dropout1 = nn.Dropout(dropout) # check
        self.pos_encoding = positional_encoding(max_seq_length, d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        seq_length = x.size(1)
        x = self.embedding(x) * math.sqrt(self.d_model)
        
        # Use only the required amount of positional encoding
        pos_encoding = self.pos_encoding[:, :seq_length, :].to(x.device)
        
        x = x + pos_encoding
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, mask)

        return x

In [3]:
import torch
import torch.nn as nn
import math
#from utils import MultiHeadAttention, PositionWiseFeedForward, positional_encoding

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.dropout = nn.Dropout(dropout)
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        
        self.dropout1 = nn.Dropout(dropout)  #check
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.dropout2 = nn.Dropout(dropout)  #check
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout)  #check
        self.norm3 = nn.LayerNorm(d_model)
        

    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attention(x, x, x, tgt_mask)
        drop = self.dropout(attn_output)
        x = self.norm1(x + drop)
        attn_output = self.cross_attention(x, enc_output, enc_output, src_mask)
        cross_drop = self.dropout(attn_output)
        x = self.norm2(x + cross_drop)
        ff_output = self.feed_forward(x)
        return  self.norm3(x + self.dropout(ff_output))
        

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout):
        super(Decoder, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.dropout1 = nn.Dropout(dropout) #check
        self.positional_encoding = positional_encoding(max_seq_length, d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        

    def forward(self, x, enc_output, src_mask, tgt_mask):
        seq_length = x.size(1)
        sqrt_d_model = math.sqrt(self.d_model)
        x = self.embedding(x) * sqrt_d_model
        pos_enc = self.positional_encoding[:, :seq_length, :].to(x.device)
        x = x + pos_enc
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)

        return x.to(device)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
#from encoder import Encoder
#from decoder import Decoder
#from utils import create_padding_mask, create_look_ahead_mask
import numpy as np
from collections import Counter
from typing import List,Tuple,Dict
import re
import nltk

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        print(f"Initializing Transformer with src_vocab_size={src_vocab_size}, tgt_vocab_size={tgt_vocab_size}")
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout)
        self.final_layer = nn.Linear(d_model, tgt_vocab_size)
        print(f"Final layer output size: {tgt_vocab_size}")
        

    def forward(self, src, tgt, src_mask, tgt_mask):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        output = self.final_layer(dec_output)
        return output
    
def load_data(file_path: str) -> List[str]:
    with open(file_path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]

def clean_text(text: str) -> str:
    # Remove special characters and digits
    clean = ""
    for txt in text:
        clean += re.sub(r'[^a-zA-ZÀ-ÿ\s]', '', txt)
    # Convert to lowercase
    clean = clean.lower()
    return clean

def tokenize_text(text: str, language: str) -> List[str]:
    cleaned_text = clean_text(text)
    if language == 'french':
        return nltk.word_tokenize(cleaned_text, language='french')
    else:  # Default to English
        return nltk.word_tokenize(cleaned_text)

def build_vocab(texts: List[str], language: str,max_vocab_size: int = 10000) -> Tuple[Dict[str, int], Dict[int, str]]:
    text = tokenize_text(texts,language)
    word_freq = Counter()
    
    if language == 'english':
        for text in texts:
            word_freq.update(tokenize_text(text, 'english'))
    else :
        for text in texts:
            word_freq.update(tokenize_text(text, 'french'))

    
    vocab = ['<pad>', '<sos>', '<eos>', '<unk>']
    vocab.extend([word for word, _ in word_freq.most_common(max_vocab_size - 4)])
    
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    
    return word2idx, idx2word

def preprocess_data(texts, word2idx, max_length):
    processed = []
    for text in texts:
        tokens = ['<sos>'] + text.split() + ['<eos>']
        ids = [word2idx.get(token, word2idx['<unk>']) for token in tokens]
        if len(ids) < max_length:
            ids += [word2idx['<pad>']] * (max_length - len(ids))
        else:
            ids = ids[:max_length-1] + [word2idx['<eos>']]
        processed.append(ids)
    return torch.LongTensor(processed)

def create_dataloaders(src_texts, tgt_texts, src_word2idx, tgt_word2idx, max_length, batch_size):
    src_data = preprocess_data(src_texts, src_word2idx, max_length)
    tgt_data = preprocess_data(tgt_texts, tgt_word2idx, max_length)
    dataset = TensorDataset(src_data, tgt_data)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

def train_model(model, train_dataloader, val_dataloader, num_epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # Assuming 0 is the padding index
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)

    model.to(device)
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for batch in train_dataloader:
            src, tgt = batch
            src, tgt = src.to(device), tgt.to(device)
            
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            
            #print(tgt_input)
            #print(tgt_output)
            src_mask = create_padding_mask(src).to(device)
            tgt_mask = create_look_ahead_mask(tgt_input.size(1)).to(device)
            
            optimizer.zero_grad()
            
            output = model(src, tgt_input, src_mask, tgt_mask)
            loss = criterion(output.view(-1, output.size(-1)), tgt_output.contiguous().view(-1))
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_dataloader)
        val_loss = evaluate_model(model, val_dataloader, criterion, device)
        
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    torch.save(model.state_dict(), 'transformer.pt')

def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            src, tgt = batch
            src, tgt = src.to(device), tgt.to(device)
            
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            
            src_mask = create_padding_mask(src).to(device)
            tgt_mask = create_look_ahead_mask(tgt_input.size(1)).to(device)
            
            output = model(src, tgt_input, src_mask, tgt_mask)
            loss = criterion(output.view(-1, output.size(-1)), tgt_output.contiguous().view(-1))
            
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

def translate(model, src_sentence, src_word2idx, tgt_idx2word, max_length, device):
    model.eval()
    src_tokens = ['<sos>'] + src_sentence.split() + ['<eos>']
    src_ids = [src_word2idx.get(token, src_word2idx['<unk>']) for token in src_tokens]
    src_tensor = torch.LongTensor(src_ids).unsqueeze(0).to(device)
    
    src_mask = create_padding_mask(src_tensor).to(device)
    
    with torch.no_grad():
        enc_output = model.encoder(src_tensor, src_mask)
        
    tgt_tensor = torch.LongTensor([[src_word2idx['<sos>']]]).to(device)
    
    output_tokens = []
    
    for _ in range(max_length - 1):  # -1 to account for <sos> token
        tgt_mask = create_look_ahead_mask(tgt_tensor.size(1)).to(device)
        
        with torch.no_grad():
            output = model.decoder(tgt_tensor, enc_output, src_mask, tgt_mask)
            output = model.final_layer(output)
        
        predicted_token_id = output[:, -1, :].argmax(dim=1)
        predicted_token = tgt_idx2word[predicted_token_id.item()]
        output_tokens.append(predicted_token)
        
        if predicted_token == '<eos>':
            break
        
        tgt_tensor = torch.cat([tgt_tensor, predicted_token_id.unsqueeze(0)], dim=1)
    
    #print(f"Debug - Output tokens: {output_tokens}")
    return ' '.join(token for token in output_tokens if token not in ['<sos>', '<eos>', '<pad>'])


In [5]:
if __name__ == '__main__':
    # Hyperparameters
    d_model = 512
    num_layers = 6 
    num_heads = 8
    d_ff = 2048
    max_seq_length = 100
    dropout = 0.2 # increased 
    batch_size = 64
    num_epochs = 10
    learning_rate = 0.0001

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    #device = 'cpu'
    home = '/kaggle/input/'
    # Load data
    train_src = load_data(home+'en-fr-iwslt-2016/train.en')
    train_tgt = load_data(home+'en-fr-iwslt-2016/train.fr')
    val_src = load_data(home+'en-fr-iwslt-2016/dev.en')
    val_tgt = load_data(home+'en-fr-iwslt-2016/dev.fr')
    test_src = load_data(home+'en-fr-iwslt-2016/test.en')
    test_tgt = load_data(home+'en-fr-iwslt-2016/test.fr')

    # Build vocabularies
    src_word2idx, src_idx2word = build_vocab(train_src,'english')
    tgt_word2idx, tgt_idx2word = build_vocab(train_tgt + val_tgt + test_tgt,'french')
    
    
    # Create dataloaders
    train_dataloader = create_dataloaders(train_src, train_tgt, src_word2idx, tgt_word2idx, max_seq_length, batch_size)
    val_dataloader = create_dataloaders(val_src, val_tgt, src_word2idx, tgt_word2idx, max_seq_length, batch_size)
    test_dataloader = create_dataloaders(test_src, test_tgt, src_word2idx, tgt_word2idx, max_seq_length, batch_size)

    # Initialize model
    model = Transformer(
        len(src_word2idx), len(tgt_word2idx), d_model, num_layers, num_heads, d_ff, max_seq_length, dropout
    )

    # Train model
    train_model(model, train_dataloader, val_dataloader, num_epochs, learning_rate, device)

    # Test model
    test_loss = evaluate_model(model, test_dataloader, nn.CrossEntropyLoss(ignore_index=0), device)
    print(f"Test Loss: {test_loss:.4f}")

    # Translate a sample sentence
    sample_sentence = "Hello, how are you?"
    translated_sentence = translate(model, sample_sentence, src_word2idx, tgt_idx2word, max_seq_length, device)
    print(f"Sample translation:")
    print(f"Input: {sample_sentence}")
    print(f"Output: {translated_sentence}") 

Initializing Transformer with src_vocab_size=10000, tgt_vocab_size=10000
value
value
value
value
value
value
value
value
value
value
value
value
value
value
value
value
value
value
Final layer output size: 10000
Epoch 1/10, Train Loss: 3.5897, Val Loss: 2.3574
Epoch 2/10, Train Loss: 1.9456, Val Loss: 1.5491
Epoch 3/10, Train Loss: 1.3620, Val Loss: 1.0698
Epoch 4/10, Train Loss: 1.0057, Val Loss: 0.8124
Epoch 5/10, Train Loss: 0.7672, Val Loss: 0.6405
Epoch 6/10, Train Loss: 0.5999, Val Loss: 0.4847
Epoch 7/10, Train Loss: 0.4714, Val Loss: 0.3925
Epoch 8/10, Train Loss: 0.3767, Val Loss: 0.3125
Epoch 9/10, Train Loss: 0.2994, Val Loss: 0.2716
Epoch 10/10, Train Loss: 0.2454, Val Loss: 0.2240
Test Loss: 0.2204
Sample translation:
Input: Hello, how are you?
Output: <unk> leur <unk> leur <unk> leur <unk> leur <unk> leur <unk> leur <unk> leur <unk>


In [6]:
def debug_translation(model, src_sentence, src_word2idx, tgt_idx2word, max_length, device):
    model.eval()
    src_tokens = ['<sos>'] + src_sentence.split() + ['<eos>']
    src_ids = [src_word2idx.get(token, src_word2idx['<unk>']) for token in src_tokens]
    src_tensor = torch.LongTensor(src_ids).unsqueeze(0).to(device)
    
    src_mask = create_padding_mask(src_tensor).to(device)
    
    with torch.no_grad():
        enc_output = model.encoder(src_tensor, src_mask)
        
    tgt_tensor = torch.LongTensor([[src_word2idx['<sos>']]]).to(device)
    
    tgt_mask = create_look_ahead_mask(tgt_tensor.size(1)).to(device)
    
    with torch.no_grad():
        output = model.decoder(tgt_tensor, enc_output, src_mask, tgt_mask)
        output = model.final_layer(output)
    
    probabilities = torch.nn.functional.softmax(output[0, 0], dim=0)
    top_5_probs, top_5_indices = torch.topk(probabilities, 5)
    
    print("Top 5 predicted tokens:")
    for prob, idx in zip(top_5_probs, top_5_indices):
        token = tgt_idx2word[idx.item()]
        print(f"{token}: {prob.item():.4f}")

# Usage
sample_sentence = "Hello"
debug_translation(model, sample_sentence, src_word2idx, tgt_idx2word, max_seq_length, device)

translated_sentence = translate(model, sample_sentence, src_word2idx, tgt_idx2word, max_seq_length, device)
print(f"Sample translation:")
print(f"Input: {sample_sentence}")
print(f"Output: {translated_sentence}")

Top 5 predicted tokens:
<unk>: 0.8590
les: 0.0303
<eos>: 0.0129
et: 0.0116
des: 0.0063
Sample translation:
Input: Hello
Output: <unk> leur <unk> leur <unk> leur <unk> leur <unk> leur <unk> leur <unk> leur <unk>


In [7]:
print(f"Source vocabulary size: {len(src_word2idx)}")
print(f"Target vocabulary size: {len(tgt_idx2word)}")

print("\nSample source word to index mappings:")
for word in list(src_word2idx.keys())[:10]:
    print(f"{word}: {src_word2idx[word]}")

print("\nSample target index to word mappings:")
for idx in list(tgt_idx2word.keys())[:10]:
    print(f"{idx}: {tgt_idx2word[idx]}")

Source vocabulary size: 10000
Target vocabulary size: 10000

Sample source word to index mappings:
<pad>: 0
<sos>: 1
<eos>: 2
<unk>: 3
the: 4
and: 5
to: 6
of: 7
a: 8
that: 9

Sample target index to word mappings:
0: <pad>
1: <sos>
2: <eos>
3: <unk>
4: de
5: et
6: la
7: les
8: le
9: que


In [8]:
import torch
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm


def evaluate_and_save(model, test_src, test_tgt, src_word2idx, tgt_idx2word, max_seq_length, device, output_file):
    model.eval()
    smooth = SmoothingFunction().method1
    total_bleu_score = 0
    num_sentences = len(test_src)

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("Source Sentence\tBLEU Score\tTranslation\tReference\n")

        for src_sentence, tgt_sentence in tqdm(zip(test_src, test_tgt), total=num_sentences, desc="Evaluating"):
            model_output = translate(model, src_sentence, src_word2idx, tgt_idx2word, max_seq_length, device)
            
            # Tokenize sentences
            reference = [tgt_sentence.split()]
            hypothesis = model_output.split()
            
            # Calculate BLEU score
            bleu_score = sentence_bleu(reference, hypothesis, smoothing_function=smooth)
            total_bleu_score += bleu_score
            
            # Write to file
            f.write(f"{src_sentence}\t{bleu_score:.4f}\t{model_output}\t{tgt_sentence}\n")

    avg_bleu_score = total_bleu_score / num_sentences
    
    # Write average BLEU score at the end of the file
    with open(output_file, 'a', encoding='utf-8') as f:
        f.write(f"\nAverage BLEU Score: {avg_bleu_score:.4f}")

    print(f'Average BLEU Score: {avg_bleu_score:.4f}')
    print(f'Results written to {output_file}')

# Usage
if __name__ == "__main__":
    # ... (your existing setup code)

    # Evaluate the model and save results
    output_file = "translation_evaluation_results.txt"
    evaluate_and_save(model, test_src, test_tgt, src_word2idx, tgt_idx2word, max_seq_length, device, output_file)

Evaluating: 100%|██████████| 1305/1305 [02:17<00:00,  9.52it/s]

Average BLEU Score: 0.0003
Results written to translation_evaluation_results.txt
